# fase_3 - script_cimut Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Benerin database lama ke database baru untuk bagian CRM, Prospek, dan Operasional.

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import datetime
import random
import string
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## Ambil Data dari DB Lama

In [3]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS
# ---------------------------------------------------------
cursor_old.execute("SHOW TABLES")
tables_data = cursor_old.fetchall()
target_tables = [list(t.values())[0] for t in tables_data]
print(f"\n--- Ditemukan {len(target_tables)} tabel di Database Lama ---")

df_old = {}
for table in target_tables:
    try:
        query = f"SELECT * FROM `{table}`"
        df_old[table] = pd.read_sql(query, db_old)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_old[table])}")
    except Exception as e:
        print(f"Gagal load tabel {table}: {e}")

print("\n--- Proses load selesai. Semua data tersimpan di 'df_old' ---")


--- Ditemukan 108 tabel di Database Lama ---
Berhasil load tabel: absensi | Jumlah baris: 13444
Berhasil load tabel: absensi_note | Jumlah baris: 11
Berhasil load tabel: bidang | Jumlah baris: 4
Berhasil load tabel: bidangkategori | Jumlah baris: 12
Berhasil load tabel: bidanglink | Jumlah baris: 7
Berhasil load tabel: calon | Jumlah baris: 4
Berhasil load tabel: calon_detil | Jumlah baris: 61
Berhasil load tabel: calon_pertanyaan | Jumlah baris: 229
Berhasil load tabel: calon_pertanyaan_detil | Jumlah baris: 4305
Berhasil load tabel: catatan_kelas | Jumlah baris: 12797
Berhasil load tabel: catatan_kelas_tag | Jumlah baris: 999
Berhasil load tabel: catatan_mingguan | Jumlah baris: 0
Berhasil load tabel: catatan_siswa | Jumlah baris: 1502
Berhasil load tabel: catatan_siswa_follow_up | Jumlah baris: 22
Berhasil load tabel: catatanawal_admin | Jumlah baris: 64
Berhasil load tabel: catatanawal_datautama | Jumlah baris: 9
Berhasil load tabel: catatanawal_infolain | Jumlah baris: 64
Berhasi

## Ambil Data dari DB Baru (Struktur Target)

In [4]:
cursor_new.execute("SHOW TABLES")
tables_data_new = cursor_new.fetchall()
target_tables_new = [list(t.values())[0] for t in tables_data_new]
df_new = {}

for table in target_tables_new:
    try:
        query = f"SELECT * FROM `{table}`"
        df_new[table] = pd.read_sql(query, db_new)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_new[table])}")
    except:
        pass

Berhasil load tabel: absensi | Jumlah baris: 0
Berhasil load tabel: activity_log | Jumlah baris: 0
Berhasil load tabel: admin_sarpras | Jumlah baris: 1
Berhasil load tabel: bidang_kategori | Jumlah baris: 12
Berhasil load tabel: bidang_link | Jumlah baris: 7
Berhasil load tabel: busdev_bidang | Jumlah baris: 4
Berhasil load tabel: cache | Jumlah baris: 0
Berhasil load tabel: cache_locks | Jumlah baris: 0
Berhasil load tabel: calon_siswa | Jumlah baris: 0
Berhasil load tabel: calon_siswa_akademik | Jumlah baris: 0
Berhasil load tabel: calon_siswa_bayar | Jumlah baris: 0
Berhasil load tabel: calon_siswa_jadwal | Jumlah baris: 0
Berhasil load tabel: calon_siswa_kursus | Jumlah baris: 0
Berhasil load tabel: calon_siswa_ortu | Jumlah baris: 0
Berhasil load tabel: calon_siswa_proses | Jumlah baris: 0
Berhasil load tabel: calon_siswa_status_logs | Jumlah baris: 0
Berhasil load tabel: catatan_kelas | Jumlah baris: 0
Berhasil load tabel: catatan_kelas_tag | Jumlah baris: 0
Berhasil load tabel: 

# Fixing kontak_prospek
Target: `kontak_prospek, calon_siswa, calon_siswa_akademik, calon_siswa_ortu, calon_siswa_bayar, calon_siswa_jadwal, calon_siswa_kursus, calon_siswa_proses, calon_siswa_status_logs, peminjaman, pengadaan, problem.`

In [5]:
display(df_new['kontak_prospek'])

,id_kontak_prospek,kode_kontak,nama_penanya,nomor_telepon,email,sumber_informasi,catatan_awal_fo,id_admin_fo,status_kontak,tanggal_kontak_pertama,tanggal_kontak_terakhir,created_at,updated_at


## Inspeksi Tabel Sumber (kontak_prospek)
Jalankan cell ini untuk melihat isi asli tabel-tabel di df_old yang berkaitan dengan kontak awal.

In [6]:
source_tables = [
    'catatanawal_datautama', 
    'catatanawal_admin', 
    'catatanawal_infolain', 
    'catatanawal_tglpenting'
]

print("--- INSPEKSI TABEL SUMBER ---")
for t in source_tables:
    if t in df_old:
        print(f"\n>> Tabel: {t} ({len(df_old[t])} baris)")
        print(f"Kolom: {df_old[t].columns.tolist()}")
        display(df_old[t].head(3))
    else:
        print(f"\n>> Tabel {t} tidak ditemukan!")

--- INSPEKSI TABEL SUMBER ---

>> Tabel: catatanawal_datautama (9 baris)
Kolom: ['idcatatanawal_datautama', 'pengontak_admin', 'nama_l', 'nama_p', 'jenis_k', 'no_wa_ortu', 'no_wa_anak', 'email', 'pilihan_program', 'jenis_program', 'level_1', 'level_2', 'tujuan_program', 'metode', 'info', 'referensi', 'sby_luarsby', 'kewarganegaraan', 'provinsi', 'kabupaten', 'kecamatan', 'kelurahan', 'alamat_lengkap', 'nama_instansi', 'kurikulum', 'pernah_les', 'kesulitan_pelajaran', 'idcatatanawal_admin']


,idcatatanawal_datautama,pengontak_admin,nama_l,nama_p,jenis_k,no_wa_ortu,no_wa_anak,email,pilihan_program,jenis_program,...,provinsi,kabupaten,kecamatan,kelurahan,alamat_lengkap,nama_instansi,kurikulum,pernah_les,kesulitan_pelajaran,idcatatanawal_admin
0,N000000001,Ibu Sari,Daria Azmiya Jasmine,Daria,Perempuan,082231346758,,puterihapsari.f@gmail.com,English,GE,...,0,0,0,0,,TK Al Maghfirah,NASIONAL,YA,,M000000002
1,N000000002,Bu Dwi,Annisa Zahro Ramadhania,Annisa,Perempuan,081336647476,,dwirohm4@gmail.com,English,GE,...,0,0,0,0,,SD Khadijah Wonorejo,CAMBRIDGE,TIDAK,YA,M000000003
2,N000000003,,Ghayda Syakira Hanania,Ghay,Perempuan,081235160064,,humaidah0208@gmail.com,English,GE,...,35,3578,0,0,,Al Amin,CAMBRIDGE,YA,YA,M000000004



>> Tabel: catatanawal_admin (64 baris)
Kolom: ['idcatatanawal_admin', 'nama', 'tlp', 'email', 'status', 'created_at', 'updated_at']


,idcatatanawal_admin,nama,tlp,email,status,created_at,updated_at
0,M000000001,Ahmad 1,0812312344531,ahmad@gmail.com,canceled,2025-09-12 00:51:16,2025-09-12 00:51:16
1,M000000002,Daria Azmiya Jasmine,082231346758,puterihapsari.f@gmail.com,done,2025-09-15 13:33:49,2025-09-18 17:39:40
2,M000000003,Annisa Zahro Ramadhania,081336647476,dwirohm4@gmail.com,done,2025-09-16 11:13:41,2025-09-23 16:25:14



>> Tabel: catatanawal_infolain (64 baris)
Kolom: ['idcatatanawal_infolain', 'jenis_test', 'keterangan', 'trial', 'hasil_test', 'wawancara', 'catatan_penting', 'diterima_dikelas', 'form_daftar', 'bulan_masuk', 'bank', 'wag_lv', 'pic', 'idcatatanawal_admin']


,idcatatanawal_infolain,jenis_test,keterangan,trial,hasil_test,wawancara,catatan_penting,diterima_dikelas,form_daftar,bulan_masuk,bank,wag_lv,pic,idcatatanawal_admin
0,P000000001,,,,,,,,Belum,,,,,M000000001
1,P000000004,,,,,,,,Belum,,,,,M000000004
2,P000000005,,,,,,,,Belum,,,,,M000000005



>> Tabel: catatanawal_tglpenting (10 baris)
Kolom: ['idcatatanawal_tglpenting', 'kontakA_date', 'wawancara_date', 'trial_date', 'tglbayar_date', 'tglmasuk_date', 'tglkeluar_date', 'idcatatanawal_admin']


,idcatatanawal_tglpenting,kontakA_date,wawancara_date,trial_date,tglbayar_date,tglmasuk_date,tglkeluar_date,idcatatanawal_admin
0,O000000007,2025-09-16,None,None,None,None,None,M000000007
1,O000000009,2025-09-15,None,2025-09-15,2025-09-17,2025-09-24,None,M000000002
2,O000000012,2025-09-15,2025-09-15,2025-09-15,None,2025-09-25,None,M000000004


## Tahap 1: Penggabungan (catatanawal_all)
Menggabungkan 4 tabel catatan awal menjadi satu tabel referensi lengkap.

In [7]:
def create_catatanawal_all():
    JOIN_KEY = 'idcatatanawal_admin'
    
    if 'catatanawal_admin' not in df_old:
        print("Error: catatanawal_admin tidak ditemukan!")
        return pd.DataFrame()

    # 1. Base Table: catatanawal_admin
    catatanawal_all = df_old['catatanawal_admin'].copy()
    
    # 2. Join dengan catatanawal_datautama
    if 'catatanawal_datautama' in df_old:
        catatanawal_all = pd.merge(catatanawal_all, df_old['catatanawal_datautama'], on=JOIN_KEY, how='left', suffixes=('', '_utama'))
    
    # 3. Join dengan catatanawal_infolain
    if 'catatanawal_infolain' in df_old:
        catatanawal_all = pd.merge(catatanawal_all, df_old['catatanawal_infolain'], on=JOIN_KEY, how='left', suffixes=('', '_info'))
        
    # 4. Join dengan catatanawal_tglpenting
    if 'catatanawal_tglpenting' in df_old:
        catatanawal_all = pd.merge(catatanawal_all, df_old['catatanawal_tglpenting'], on=JOIN_KEY, how='left', suffixes=('', '_tgl'))

    return catatanawal_all

# Buat tabel gabungan
catatanawal_all = create_catatanawal_all()

print(f"--- Tabel 'catatanawal_all' Berhasil Dibuat ---")
print(f"Total Baris: {len(catatanawal_all)}")
print(f"Total Kolom: {len(catatanawal_all.columns)}")
display(catatanawal_all.head())
print("\nKolom yang tersedia:")
print(catatanawal_all.columns.tolist())

--- Tabel 'catatanawal_all' Berhasil Dibuat ---
Total Baris: 64
Total Kolom: 54


,idcatatanawal_admin,nama,tlp,email,status,created_at,updated_at,idcatatanawal_datautama,pengontak_admin,nama_l,...,bank,wag_lv,pic,idcatatanawal_tglpenting,kontakA_date,wawancara_date,trial_date,tglbayar_date,tglmasuk_date,tglkeluar_date
0,M000000001,Ahmad 1,0812312344531,ahmad@gmail.com,canceled,2025-09-12 00:51:16,2025-09-12 00:51:16,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000000002,Daria Azmiya Jasmine,082231346758,puterihapsari.f@gmail.com,done,2025-09-15 13:33:49,2025-09-18 17:39:40,N000000001,Ibu Sari,Daria Azmiya Jasmine,...,Transfer Bank,,,O000000009,2025-09-15,None,2025-09-15,2025-09-17,2025-09-24,None
2,M000000003,Annisa Zahro Ramadhania,081336647476,dwirohm4@gmail.com,done,2025-09-16 11:13:41,2025-09-23 16:25:14,N000000002,Bu Dwi,Annisa Zahro Ramadhania,...,,,,O000000014,2025-09-16,2025-09-16,2025-09-16,None,2025-09-25,None
3,M000000004,Ghayda Syakira Hanania,081235160064,humaidah0208@gmail.com,done,2025-09-17 16:35:22,2025-10-03 09:47:53,N000000003,,Ghayda Syakira Hanania,...,,,,O000000012,2025-09-15,2025-09-15,2025-09-15,None,2025-09-25,None
4,M000000005,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,done,2025-09-17 16:49:20,2025-10-03 09:49:15,N000000004,Bu Lita,Achmad Naufal Albiruni,...,,,,O000000013,2025-09-17,2025-09-17,2025-09-17,None,2025-09-25,None



Kolom yang tersedia:
['idcatatanawal_admin', 'nama', 'tlp', 'email', 'status', 'created_at', 'updated_at', 'idcatatanawal_datautama', 'pengontak_admin', 'nama_l', 'nama_p', 'jenis_k', 'no_wa_ortu', 'no_wa_anak', 'email_utama', 'pilihan_program', 'jenis_program', 'level_1', 'level_2', 'tujuan_program', 'metode', 'info', 'referensi', 'sby_luarsby', 'kewarganegaraan', 'provinsi', 'kabupaten', 'kecamatan', 'kelurahan', 'alamat_lengkap', 'nama_instansi', 'kurikulum', 'pernah_les', 'kesulitan_pelajaran', 'idcatatanawal_infolain', 'jenis_test', 'keterangan', 'trial', 'hasil_test', 'wawancara', 'catatan_penting', 'diterima_dikelas', 'form_daftar', 'bulan_masuk', 'bank', 'wag_lv', 'pic', 'idcatatanawal_tglpenting', 'kontakA_date', 'wawancara_date', 'trial_date', 'tglbayar_date', 'tglmasuk_date', 'tglkeluar_date']


## Tahap 2: Transformasi ke df_new (kontak_prospek)
Mencocokkan kolom dari catatanawal_all ke struktur target kontak_prospek.

In [8]:
def generate_random_code(length=10):
    """Membuat captcha acak campuran huruf dan angka."""
    characters = string.ascii_uppercase + string.digits
    return ''.join(random.choice(characters) for i in range(length))

def transform_to_kontak_prospek(df_source):
    final_data = []
    
    for i, row in df_source.iterrows():
        # Cleaning & Logic
        telp_raw = row.get('tlp') or row.get('no_wa_ortu') or ''
        clean_phone = ''.join(filter(str.isdigit, str(telp_raw)))
        
        nama_final = row.get('nama_l') or row.get('nama') or 'Tanpa Nama'
        
        # Mapping ke df_new['kontak_prospek']
        record = {
            'id_kontak_prospek': row.get('idcatatanawal_admin'),
            'kode_kontak': generate_random_code(10), # Captcha acak 10 karakter
            'nama_penanya': nama_final,
            'nomor_telepon': clean_phone,
            'email': row.get('email'),
            'sumber_informasi': row.get('info') or row.get('sumber') or '',
            'catatan_awal_fo': row.get('catatan') or '',
            'id_admin_fo': row.get('pengontak_admin') or row.get('admin') or '',
            'status_kontak': row.get('status') or '',
            'tanggal_kontak_pertama': row.get('created_at'),
            'tanggal_kontak_terakhir': row.get('updated_at'),
            'created_at': row.get('created_at', datetime.datetime.now()),
            'updated_at': datetime.datetime.now()
        }
        final_data.append(record)
    
    return pd.DataFrame(final_data)

# Jalankan Transformasi
df_kontak_prospek_ready = transform_to_kontak_prospek(catatanawal_all)

print("\n--- HASIL PENYUSUNAN KONTAK_PROSPEK (SIAP INSERT) ---")
display(df_kontak_prospek_ready)


--- HASIL PENYUSUNAN KONTAK_PROSPEK (SIAP INSERT) ---


,id_kontak_prospek,kode_kontak,nama_penanya,nomor_telepon,email,sumber_informasi,catatan_awal_fo,id_admin_fo,status_kontak,tanggal_kontak_pertama,tanggal_kontak_terakhir,created_at,updated_at
0,M000000001,75XV66P6DE,NaN,0812312344531,ahmad@gmail.com,NaN,,NaN,canceled,2025-09-12 00:51:16,2025-09-12 00:51:16,2025-09-12 00:51:16,2026-05-12 13:09:42.108067
1,M000000002,LRHXRTYVMO,Daria Azmiya Jasmine,082231346758,puterihapsari.f@gmail.com,Teman/Saudara,,Ibu Sari,done,2025-09-15 13:33:49,2025-09-18 17:39:40,2025-09-15 13:33:49,2026-05-12 13:09:42.108140
2,M000000003,Y60LZNO5WH,Annisa Zahro Ramadhania,081336647476,dwirohm4@gmail.com,Teman/Saudara,,Bu Dwi,done,2025-09-16 11:13:41,2025-09-23 16:25:14,2025-09-16 11:13:41,2026-05-12 13:09:42.108195
3,M000000004,ETNI5KNXQB,Ghayda Syakira Hanania,081235160064,humaidah0208@gmail.com,Teman/Saudara,,,done,2025-09-17 16:35:22,2025-10-03 09:47:53,2025-09-17 16:35:22,2026-05-12 13:09:42.108248
4,M000000005,P4LLXO84II,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,,,Bu Lita,done,2025-09-17 16:49:20,2025-10-03 09:49:15,2025-09-17 16:49:20,2026-05-12 13:09:42.108295
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,M000000060,27DYQH6S4C,NaN,0811537168,cheryl2013sby@gmail.com,NaN,,NaN,on progress,2025-10-24 15:23:53,2025-10-24 15:23:53,2025-10-24 15:23:53,2026-05-12 13:09:42.110910
60,M000000061,O07YW9F8YO,NaN,08113322866,asihasih180808@gmail.com,NaN,,NaN,on progress,2025-10-24 15:24:48,2025-10-24 15:24:48,2025-10-24 15:24:48,2026-05-12 13:09:42.110950
61,M000000062,FAO42LWZC4,NaN,082264644495,,NaN,,NaN,follow up another time,2025-10-24 15:27:54,2025-10-24 15:27:54,2025-10-24 15:27:54,2026-05-12 13:09:42.110988
62,M000000063,R6TSIFZ7IS,NaN,0895368241226,,NaN,,NaN,waiting for confirmation,2025-10-24 15:28:21,2025-10-24 15:28:21,2025-10-24 15:28:21,2026-05-12 13:09:42.111027


# Migrasi Calon Siswa (Step-by-Step)

## Tahap 1: Penggabungan Data (Merging)
Menyatukan `form_calon`, `form_calon_detil1-4`, dan `catatanawal_all` menjadi satu DataFrame master.

In [25]:
display(df_new['calon_siswa'].info())
# display(df_new['calon_siswa_akademik'].info())
# display(df_new['calon_siswa_bayar'].info())
# display(df_new['calon_siswa_jadwal'].info())
# display(df_new['calon_siswa_kursus'].info())
# display(df_new['calon_siswa_ortu'].info())
# display(df_new['calon_siswa_proses'].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 31 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_calon           0 non-null      object
 1   kode_unik          0 non-null      object
 2   nama_lengkap       0 non-null      object
 3   id_kontak_prospek  0 non-null      object
 4   nama_panggilan     0 non-null      object
 5   jenis_kelamin      0 non-null      object
 6   tempat_lahir       0 non-null      object
 7   tanggal_lahir      0 non-null      object
 8   kewarganegaraan    0 non-null      object
 9   email              0 non-null      object
 10  nama_kontak_awal   0 non-null      object
 11  wa_kontak_awal     0 non-null      object
 12  id_provinsi        0 non-null      object
 13  id_kabupaten       0 non-null      object
 14  id_kecamatan       0 non-null      object
 15  id_kelurahan       0 non-null      object
 16  alamat_lengkap     0 non-null      object
 17  wa_siswa 

None

In [22]:
display(df_old['form_calon'])
# display(df_old['form_calon_detil1'])
# display(df_old['form_calon_detil2'])
# display(df_old['form_calon_detil3'])
# display(df_old['form_calon_detil4'])

,idcalon,fullName,email,nickName,phone1,phone2,schoolName,classLevel,gender,activities,...,date,file,kewarganegaraan,provinsi,kabupaten,status,keterangan,catatanadmin,idpendkursus,created_at
0,C00000017,Daria Azmiya Jasmine,puterihapsari.f@gmail.com,Daria,082231346758,,TK Al Maghfirah,TK B,Perempuan,None,...,None,None,Indonesia,35,3578.0,1,None,None,K00010,2025-09-15 11:29:55
1,C00000018,Annisa Zahro Ramadhania,dwirohm4@gmail.com,Annisa,081336647476,,Sd Khadijah wonorejo,Kelas 3,Perempuan,None,...,None,None,Indonesia,35,3578.0,1,None,None,K00010,2025-09-16 10:48:00
2,C00000019,Zulfa Bariatur Rahma,chyzryth@gmail.com,Zulfa,085707179656,083849241708,SMAN 17 Surabaya,X,None,None,...,"Kamis, Pukul 17.00 - 18.00 WIB",None,Indonesia,35,3578.0,1,None,None,K00014,2025-09-17 11:34:19
3,C00000020,Achmad Naufal Albiruni,melisnifuku@gmail.com,AlBI,087765283592,None,SD Khadijah Wonorejo Surabaya,SD kelas 5,Laki-laki,None,...,None,None,Indonesia,35,3578.0,1,None,None,K00010,2025-09-17 12:33:32
4,C00000021,Khansa Amalia Putri Aji,afadhilpa@gmail.com,Khansa,081554932188,,SMAN 17 Surabaya,XI,Perempuan,None,...,None,None,Indonesia,35,3578.0,0,None,None,K00010,2025-09-17 13:11:27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,C00000197,Antonius Miguel Kurniawan,adeodatus.kurniawan@gmail.com,Miguel,08170326911,None,SD Kartika Nasional Plus,SD Kelas 2,Laki-laki,None,...,None,None,Indonesia,35,3578.0,0,None,None,K00010,2026-04-02 13:20:19
180,C00000198,Tisha kayla janitra,nroskalindha18@gmail.com,Tisha,082244441630,None,Tk al fajar,Tk b,Perempuan,None,...,None,None,Indonesia,35,3578.0,1,None,None,K00010,2026-04-13 15:58:41
181,C00000199,Clariza Arifianti,clarizarisa3@gmail.com,Clara,08977257033,None,None,None,Perempuan,Mahasiswa,...,None,None,Indonesia,35,3578.0,0,None,None,K00004,2026-04-13 17:28:08
182,C00000200,Shaqila Anindra Dzakira,shaqilaanindra@gmail.com,Shaqila,085850209079,None,SDIT Ghilmani Surabaya,kelas 4,Perempuan,None,...,None,None,Indonesia,35,3578.0,1,None,None,K00010,2026-04-14 07:20:01


In [17]:
# 1. Ambil base pendaftaran
df_calon_master = df_old['form_calon'].copy()

# 2. Join dengan detail tambahan (detil1 - detil4)
for i in range(1, 5):
    table_name = f'form_calon_detil{i}'
    if table_name in df_old:
        df_calon_master = pd.merge(df_calon_master, df_old[table_name], on='idcalon', how='left', suffixes=('', f'_d{i}'))

# 3. Join dengan catatan awal (jika ada variable catatanawal_all dari cell sebelumnya)
if 'catatanawal_all' in locals() or 'catatanawal_all' in globals():
    df_calon_master = pd.merge(df_calon_master, catatanawal_all, 
                               left_on='idpendkursus', right_on='idcatatanawal_admin', 
                               how='left', suffixes=('', '_ca'))

print(f"--- Master Dataframe 'df_calon_master' Berhasil Dibuat ---")
print(f"Total Baris: {len(df_calon_master)}")
display(df_calon_master)

--- Master Dataframe 'df_calon_master' Berhasil Dibuat ---
Total Baris: 184


,idcalon,fullName,email,nickName,phone1,phone2,schoolName,classLevel,gender,activities,...,bank_ca,wag_lv,pic,idcatatanawal_tglpenting,kontakA_date,wawancara_date,trial_date,tglbayar_date,tglmasuk_date,tglkeluar_date
0,C00000017,Daria Azmiya Jasmine,puterihapsari.f@gmail.com,Daria,082231346758,,TK Al Maghfirah,TK B,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C00000018,Annisa Zahro Ramadhania,dwirohm4@gmail.com,Annisa,081336647476,,Sd Khadijah wonorejo,Kelas 3,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C00000019,Zulfa Bariatur Rahma,chyzryth@gmail.com,Zulfa,085707179656,083849241708,SMAN 17 Surabaya,X,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C00000020,Achmad Naufal Albiruni,melisnifuku@gmail.com,AlBI,087765283592,None,SD Khadijah Wonorejo Surabaya,SD kelas 5,Laki-laki,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C00000021,Khansa Amalia Putri Aji,afadhilpa@gmail.com,Khansa,081554932188,,SMAN 17 Surabaya,XI,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,C00000197,Antonius Miguel Kurniawan,adeodatus.kurniawan@gmail.com,Miguel,08170326911,None,SD Kartika Nasional Plus,SD Kelas 2,Laki-laki,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
180,C00000198,Tisha kayla janitra,nroskalindha18@gmail.com,Tisha,082244441630,None,Tk al fajar,Tk b,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,C00000199,Clariza Arifianti,clarizarisa3@gmail.com,Clara,08977257033,None,None,None,Perempuan,Mahasiswa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
182,C00000200,Shaqila Anindra Dzakira,shaqilaanindra@gmail.com,Shaqila,085850209079,None,SDIT Ghilmani Surabaya,kelas 4,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Tahap 2: Pembersihan & Standarisasi
Melakukan cleaning pada nama, nomor WA, gender, dan status pipeline.

In [28]:
def clean_phone(phone):
    if pd.isna(phone) or str(phone).strip() == '': return None
    phone = ''.join(filter(str.isdigit, str(phone)))
    if phone.startswith('0'): phone = '62' + phone[1:]
    if not phone.startswith('62') and phone != '': phone = '62' + phone
    return phone[:15]

# 1. Cleaning Dasar
df_calon_master['nama_lengkap'] = df_calon_master['fullName'].str.strip().str.title()
df_calon_master['nama_panggilan'] = df_calon_master['nickName'].str.strip().str.title()

# 2. Standarisasi WA
df_calon_master['wa_siswa_clean'] = df_calon_master['phone1'].apply(clean_phone)
df_calon_master['wa_ortu_clean'] = df_calon_master['phone2'].apply(clean_phone)

# 3. Mapping Gender
gender_map = {
    'Laki-laki': 'Laki laki', 'Laki-Laki': 'Laki laki', 'L': 'Laki laki', 'Pria': 'Laki laki',
    'Perempuan': 'Perempuan', 'Wanita': 'Perempuan', 'P': 'Perempuan'
}

print("--- Data Cleansing Selesai ---")
df_calon_master

--- Data Cleansing Selesai ---


,idcalon,fullName,email,nickName,phone1,phone2,schoolName,classLevel,gender,activities,...,tglbayar_date,tglmasuk_date,tglkeluar_date,nama_lengkap,nama_panggilan,wa_siswa_clean,wa_ortu_clean,jenis_kelamin_clean,status_pipeline_clean,jenis_kelamin
0,C00000017,Daria Azmiya Jasmine,puterihapsari.f@gmail.com,Daria,082231346758,,TK Al Maghfirah,TK B,Perempuan,None,...,NaN,NaN,NaN,Daria Azmiya Jasmine,Daria,6282231346758,None,Perempuan,proses,Perempuan
1,C00000018,Annisa Zahro Ramadhania,dwirohm4@gmail.com,Annisa,081336647476,,Sd Khadijah wonorejo,Kelas 3,Perempuan,None,...,NaN,NaN,NaN,Annisa Zahro Ramadhania,Annisa,6281336647476,None,Perempuan,proses,Perempuan
2,C00000019,Zulfa Bariatur Rahma,chyzryth@gmail.com,Zulfa,085707179656,083849241708,SMAN 17 Surabaya,X,None,None,...,NaN,NaN,NaN,Zulfa Bariatur Rahma,Zulfa,6285707179656,6283849241708,Laki laki,proses,Laki laki
3,C00000020,Achmad Naufal Albiruni,melisnifuku@gmail.com,AlBI,087765283592,None,SD Khadijah Wonorejo Surabaya,SD kelas 5,Laki-laki,None,...,NaN,NaN,NaN,Achmad Naufal Albiruni,Albi,6287765283592,None,Laki laki,proses,Laki laki
4,C00000021,Khansa Amalia Putri Aji,afadhilpa@gmail.com,Khansa,081554932188,,SMAN 17 Surabaya,XI,Perempuan,None,...,NaN,NaN,NaN,Khansa Amalia Putri Aji,Khansa,6281554932188,None,Perempuan,baru,Perempuan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,C00000197,Antonius Miguel Kurniawan,adeodatus.kurniawan@gmail.com,Miguel,08170326911,None,SD Kartika Nasional Plus,SD Kelas 2,Laki-laki,None,...,NaN,NaN,NaN,Antonius Miguel Kurniawan,Miguel,628170326911,None,Laki laki,baru,Laki laki
180,C00000198,Tisha kayla janitra,nroskalindha18@gmail.com,Tisha,082244441630,None,Tk al fajar,Tk b,Perempuan,None,...,NaN,NaN,NaN,Tisha Kayla Janitra,Tisha,6282244441630,None,Perempuan,proses,Perempuan
181,C00000199,Clariza Arifianti,clarizarisa3@gmail.com,Clara,08977257033,None,None,None,Perempuan,Mahasiswa,...,NaN,NaN,NaN,Clariza Arifianti,Clara,628977257033,None,Perempuan,baru,Perempuan
182,C00000200,Shaqila Anindra Dzakira,shaqilaanindra@gmail.com,Shaqila,085850209079,None,SDIT Ghilmani Surabaya,kelas 4,Perempuan,None,...,NaN,NaN,NaN,Shaqila Anindra Dzakira,Shaqila,6285850209079,None,Perempuan,proses,Perempuan


## Tahap 3: Validasi Foreign Key (Wilayah)
Memastikan ID Provinsi dan Kabupaten ada di DB baru untuk menghindari error constraint.

In [14]:
# Ambil daftar ID valid dari DB Baru
cursor_new.execute("SELECT id_provinsi FROM provinsi")
valid_prov = [r['id_provinsi'] for r in cursor_new.fetchall()]

cursor_new.execute("SELECT id_kabupaten FROM kabupaten")
valid_kab = [r['id_kabupaten'] for r in cursor_new.fetchall()]

# Fungsi filter ID
def validate_id(val, valid_list):
    try:
        v = int(float(val))
        return v if v in valid_list else None
    except: return None

df_calon_master['id_provinsi_safe'] = df_calon_master['provinsi'].apply(lambda x: validate_id(x, valid_prov))
df_calon_master['id_kabupaten_safe'] = df_calon_master['kabupaten'].apply(lambda x: validate_id(x, valid_kab))

print(f"ID Provinsi valid ditemukan: {df_calon_master['id_provinsi_safe'].notna().sum()}")
print(f"ID Kabupaten valid ditemukan: {df_calon_master['id_kabupaten_safe'].notna().sum()}")

ID Provinsi valid ditemukan: 177
ID Kabupaten valid ditemukan: 0


## Tahap 4: Eksekusi Migrasi (Looping Insertion)
Melakukan insert berurutan ke parent table dan sub-tabel.

In [ ]:
# from datetime import datetime

# count_success = 0
# for index, row in df_calon_master.iterrows():
#     try:
#         # A. Insert CALON_SISWA (Parent)
#         kode_unik = f"CS-{datetime.now().year}-{row['idcalon']}"
        
#         sql_parent = """INSERT INTO calon_siswa 
#                      (kode_unik, nama_lengkap, nama_panggilan, jenis_kelamin, email, 
#                       wa_siswa, wa_ortu, id_provinsi, id_kabupaten, status_pipeline, created_at) 
#                      VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""
        
#         val_parent = (
#             kode_unik, row['nama_lengkap'], row['nama_panggilan'], row['jenis_kelamin_clean'], row['email'],
#             row['wa_siswa_clean'], row['wa_ortu_clean'], row['id_provinsi_safe'], row['id_kabupaten_safe'], 
#             row['status_pipeline_clean'], row['created_at']
#         )
        
#         cursor_new.execute(sql_parent, val_parent)
#         new_id = cursor_new.lastrowid 
        
#         # B. Insert CALON_SISWA_AKADEMIK
#         sql_akad = """INSERT INTO calon_siswa_akademik 
#                     (id_calon, nama_sekolah, jenjang_kelas_1, kurikulum_sekolah, riwayat_les, kesulitan_belajar) 
#                     VALUES (%s, %s, %s, %s, %s, %s)"""
#         kur_map = {'NASIONAL': 'NASIONAL', 'CAMBRIDGE': 'CAMBRIDGE'}
#         kur_val = kur_map.get(str(row.get('curriculum')).upper(), 'LAINNYA')
        
#         val_akad = (new_id, row.get('schoolName'), row.get('classLevel'), kur_val, row.get('exp'), row.get('diagnostic'))
#         cursor_new.execute(sql_akad, val_akad)
        
#         # C. Insert CALON_SISWA_ORTU
#         sql_ortu = "INSERT INTO calon_siswa_ortu (id_calon, nama_ibu, pekerjaan_ibu) VALUES (%s, %s, %s)"
#         val_ortu = (new_id, row.get('nama_ortu'), row.get('pekerjaan_ortu'))
#         cursor_new.execute(sql_ortu, val_ortu)
        
#         # D. Insert CALON_SISWA_BAYAR (Jika ada data pembayaran)
#         if pd.notna(row.get('nomor_invoice')):
#             sql_bayar = """INSERT INTO calon_siswa_bayar 
#                          (id_calon, nomor_invoice, bank_pembayaran, tanggal_konfirmasi_bayar, lokasi_belajar) 
#                          VALUES (%s, %s, %s, %s, %s)"""
#             val_bayar = (new_id, row['nomor_invoice'], row.get('bank'), row.get('tanggal_pembayaran'), row.get('lokasi'))
#             cursor_new.execute(sql_bayar, val_bayar)

#         # E. Insert CALON_SISWA_PROSES
#         sql_proses = """INSERT INTO calon_siswa_proses 
#                       (id_calon, penanggung_jawab, tanggal_trial, laporan_trial, status_diterima, followup_1) 
#                       VALUES (%s, %s, %s, %s, %s, %s)"""
#         val_proses = (new_id, row.get('PIC'), row.get('tgl_trial'), row.get('laporan_trial'), 
#                       1 if row.get('diterima') == 'YA' else 0, row.get('followUp1'))
#         cursor_new.execute(sql_proses, val_proses)
        
#         count_success += 1
#     except Exception as e:
#         print(f"Gagal baris {row['idcalon']}: {e}")
#         db_new.rollback()
#         continue

# db_new.commit()
# print(f"\n--- MIGRASI SELESAI ---")
# print(f"Berhasil: {count_success} | Total: {len(df_calon_master)}")

Gagal baris C00000017: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000018: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000019: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000021: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000022: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000023: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000024: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000025: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000032: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000033: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000034: 1265 (01000): Data truncated for column 'pekerjaan_ibu' at row 1
Gagal baris C00000035: 1265 (010

## Verifikasi Hasil

In [ ]:
# cursor_new.execute("SELECT COUNT(*) as total FROM calon_siswa")
# print(f"Total Calon Siswa di DB Baru: {cursor_new.fetchone()['total']}")

# cursor_new.execute("SELECT * FROM calon_siswa ORDER BY id_calon DESC LIMIT 5")
# pd.DataFrame(cursor_new.fetchall())

Total Calon Siswa di DB Baru: 184


,id_calon,kode_unik,nama_lengkap,id_kontak_prospek,nama_panggilan,jenis_kelamin,tempat_lahir,tanggal_lahir,kewarganegaraan,email,...,status_pipeline,status_updated_at,assigned_fo,assigned_akademik,catatan_awal_fo,link_form_sent_at,form_completed_at,deleted_at,created_at,updated_at
0,184,CS-2026-C00000201,Mega Dewi,None,Mega,Perempuan,None,None,None,megads286@gmail.com,...,baru,None,None,None,None,None,None,None,2026-04-15 16:34:23,None
1,183,CS-2026-C00000200,Shaqila Anindra Dzakira,None,Shaqila,Perempuan,None,None,None,shaqilaanindra@gmail.com,...,proses,None,None,None,None,None,None,None,2026-04-14 07:20:01,None
2,182,CS-2026-C00000199,Clariza Arifianti,None,Clara,Perempuan,None,None,None,clarizarisa3@gmail.com,...,baru,None,None,None,None,None,None,None,2026-04-13 17:28:08,None
3,181,CS-2026-C00000198,Tisha Kayla Janitra,None,Tisha,Perempuan,None,None,None,nroskalindha18@gmail.com,...,proses,None,None,None,None,None,None,None,2026-04-13 15:58:41,None
4,180,CS-2026-C00000197,Antonius Miguel Kurniawan,None,Miguel,Laki laki,None,None,None,adeodatus.kurniawan@gmail.com,...,baru,None,None,None,None,None,None,None,2026-04-02 13:20:19,None
